In [1]:
import mo_gymnasium as mo_gym
import numpy as np
import envs
import csv
from src.modules.commun import Constant
from src.modules.Tools import Tools
#from src.scripts.delete_wandb_dir import delete_directory
from morl_baselines.multi_policy.multi_policy_moqlearning.mp_mo_q_learning import (
    MPMOQLearning,
)
from src.modules.MultiCloud import MultiCloud



# IMPORTANT____________________________________________________________________________________________
# Init Env by giving it the service querry to optimize + the folder where to find the PREPROCESSED data 
serviceQuerry = [0 , 7 , 14]
number_clouds = 5
MultiCloud_data_dir=f"./src/data/preprocessedData/NC_{number_clouds}_NS_50/NC_{number_clouds}_NS_50_01"
#______________________________________________________________________________________________________


GAMMA = 1
env = mo_gym.MORecordEpisodeStatistics(mo_gym.make("env/SelectService-mpmoql", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceQuerry ), gamma=GAMMA)
eval_env = mo_gym.make("env/SelectService-mpmoql", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceQuerry )

print("done first part")




 ./src/data/preprocessedData/NC_5_NS_50/NC_5_NS_50_01 




 ./src/data/preprocessedData/NC_5_NS_50/NC_5_NS_50_01 


done first part


c:\Users\jn_fe\anaconda3\lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float64
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


#### Test env with random actions just to make sure there is no bug in env and the MDP works fine

In [2]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = env.action_space.sample()  # this is where you would insert your policy
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

#print("Selected services : ", env.composition.print_composition())

init obs: service [ [0] cloud  [0] ]
old obs: [ service [0] cloud [0] ]
action : 2 
new obs: [ service [0] cloud [4] ]
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs: [ service [0] cloud [4] ]
action : 2 
new obs: [ service [0] cloud [4] ]
reward : [-1 -1 -1 -1 -1 -1] 	erminated : True
_____________________________________________________
acc_rew: [-1 -1 -1 -1 -1 -1]


Calculate Pareto

In [3]:
pf = env.unwrapped.pareto_front()
print(pf)

Pareto already exist for [0, 7, 14] in multi cloud of 5 cloud : False


 ./src/data/preprocessedData/NC_5_NS_50/NC_5_NS_50_01 


n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |      125 |     45 |             - |             -
     2 |      125 |     45 |  0.000000E+00 |             f
The solutions are : 
Pareto front solutions:
[[0 0 1]
 [1 3 4]
 [4 3 3]
 [1 1 3]
 [2 3 0]
 [3 2 2]
 [2 3 3]
 [1 3 1]
 [4 0 0]
 [0 0 4]
 [4 4 4]
 [2 0 3]
 [0 3 3]
 [2 3 1]
 [1 0 0]
 [1 0 3]
 [3 3 3]
 [2 0 4]
 [1 0 2]
 [0 0 2]
 [0 2 2]
 [2 2 3]
 [3 2 4]
 [0 0 3]
 [1 1 1]
 [3 0 3]
 [1 3 2]
 [0 3 0]
 [2 2 4]
 [2 2 2]
 [1 3 3]
 [2 0 0]
 [1 1 2]
 [3 3 4]
 [2 3 2]
 [3 0 4]
 [2 1 1]
 [2 2 0]
 [1 3 0]
 [2 0 2]
 [2 3 4]
 [1 0 1]
 [0 0 0]
 [3 0 0]
 [2 2 1]]
Objective values of the Pareto front solutions:
[[1.4794 2.3997 0.867  2.188  1.8011 2.    ]
 [1.1183 1.777  1.9515 1.4839 2.0944 1.    ]
 [1.5475 1.4547 1.74   1.3157 2.1716 2.    ]
 [1.2326 1.9563 1.4474 1.553  2.336  2.    ]
 [1.0859 1.4096

# Init Algorithme : 

In [4]:
mp_moql = MPMOQLearning(
        env,
        learning_rate=0.3,#0.3
        gamma=GAMMA,
        use_gpi_policy=True,
        dyna=True,
        dyna_updates=5,
        initial_epsilon=1,
        final_epsilon=0.01,
        epsilon_decay_steps=int(2e5),#int(2e5)
        weight_selection_algo="ols",
        epsilon_ols=0.0,
        project_name="just testing",
        experiment_name="MPMOQL_s0s1s2",
        seed = 17)



wandb: WARNING WANDB_NOTEBOOK_NAME should be a path to a notebook file, couldn't find envelope_minecart.
wandb: Currently logged in as: nabilachehlafekir. Use `wandb login --relogin` to force relogin


# Training Agent: 

In [5]:
front = mp_moql.train(
        total_timesteps=10000,
        eval_env=eval_env,
        ref_point=np.full(Constant.number_objectives, -1),
        timesteps_per_iteration = int(1000),
        known_pareto_front=pf,
        eval_freq=10,
    )


CCS: [] CCS size: 0
Next weight: [1. 0. 0. 0. 0. 0.]
Adding value: [2.0253 2.323  0.6872 2.4381 1.8058 3.    ] to CCS.
W_corner: [array([0., 1., 0., 0., 0., 0.]), array([0., 0., 1., 0., 0., 0.]), array([0., 0., 0., 1., 0., 0.]), array([0., 0., 0., 0., 1., 0.]), array([0., 0., 0., 0., 0., 1.]), array([1., 0., 0., 0., 0., 0.])] W_corner size: 6
CCS: [array([2.0253, 2.323 , 0.6872, 2.4381, 1.8058, 3.    ], dtype=float32)] CCS size: 1
Next weight: [0. 1. 0. 0. 0. 0.]
Adding value: [1.8588 2.602  1.5506 2.5545 1.3847 2.    ] to CCS.
W_corner: [array([0.6263, 0.3737, 0.    , 0.    , 0.    , 0.    ]), array([0.4115, 0.    , 0.    , 0.5885, 0.    , 0.    ]), array([0.8383, 0.    , 0.1617, 0.    , 0.    , 0.    ]), array([1., 0., 0., 0., 0., 0.]), array([0., 0., 1., 0., 0., 0.]), array([0., 0., 0., 1., 0., 0.]), array([0., 1., 0., 0., 0., 0.]), array([0., 0., 0., 0., 1., 0.]), array([0., 0., 0., 0., 0., 1.]), array([0.    , 0.    , 0.3278, 0.    , 0.6722, 0.    ]), array([0.    , 0.    , 0.5367

charts_0/epsilon,▁
charts_1/SPS,▁
charts_1/episode_time,▁
charts_1/epsilon,▁
charts_1/timesteps_per_episode,▁
charts_2/epsilon,▁
charts_3/SPS,▁
charts_3/episode_time,▁
charts_3/epsilon,▁
charts_3/timesteps_per_episode,▁
charts_4/epsilon,▁


# Use the trained agent with diffrent prefrences  :

In [6]:

# First prefrences :
nb_clouds  = env.multicloud.getNumberClouds() 
obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/6,1/6,1/6,1/6,1/6,1/6])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())


init obs: service [ [0] cloud  [0] ]
old obs: [ service [0] cloud [0] ]
action : 3 
new obs: [ service [1] cloud [0] ]
reward : [0.9102 0.7262 0.3084 0.8233 0.2042 0.    ] 	erminated : False
_____________________________________________________
old obs: [ service [1] cloud [0] ]
action : 3 
new obs: [ service [2] cloud [0] ]
reward : [0.5101 0.9862 0.3303 0.9168 0.6934 0.    ] 	erminated : False
_____________________________________________________
old obs: [ service [2] cloud [0] ]
action : 1 
new obs: [ service [2] cloud [2] ]
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs: [ service [2] cloud [2] ]
action : 3 
new obs: [ service [3] cloud [0] ]
reward : [0.5754 0.7056 0.8103 0.8152 0.6184 2.    ] 	erminated : True
_____________________________________________________
acc_rew: [1.9957 2.418  1.449  2.5553 1.516  2.    ]


AttributeError: 'Composition' object has no attribute 'print_composition'

In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/4,1/4,1/4,0,1/4,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())

In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/2,1/4,1/4,0,0,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())